In [62]:
from dotenv import load_dotenv

load_dotenv()

True

In [63]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    ForeignKey,
    DateTime
)
from sqlalchemy.orm import relationship, sessionmaker, DeclarativeBase
from datetime import datetime

# 创建 Table 基础模型类型
class Base(DeclarativeBase):
    pass

# 创建表格 Model
class Customer(Base):
    __tablename__ = "customers"

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    
    orders = relationship("Order", back_populates="customer")

class FoodItem(Base):
    __tablename__ = "food_items"

    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    price = Column(Float, nullable=False)
    
    orders = relationship("Order", back_populates="food_item")

class Order(Base):
    __tablename__ = "orders"
    
    id = Column(Integer, primary_key=True)
    customer_id = Column(Integer, ForeignKey("customers.id"), nullable=False)
    food_item_id = Column(Integer, ForeignKey("food_items.id"), nullable=False)
    order_date = Column(DateTime, default=datetime.now)
    delivery_address = Column(String, nullable=False)

    customer = relationship("Customer", back_populates="orders")
    food_item = relationship("FoodItem", back_populates="orders")

# 创建数据库连接
engine = create_engine("sqlite:///customer.db")
Base.metadata.create_all(engine)

# 创建会话
Session = sessionmaker(bind=engine)
session = Session()

# 添加示例数据
# customer = Customer(name="John Doe")
# pizza1 = FoodItem(name="Pizza Margherita", price=8.5)
# pizza2 = FoodItem(name="Pizza Salami", price=9.5)
# pizza3 = FoodItem(name="Pizza Quattro Formaggi", price=10.50)
# session.add_all([customer, pizza1, pizza2, pizza3])
# session.commit()

# 查询数据
customers = session.query(Customer).all()
food_items = session.query(FoodItem).all()

for customer in customers:
    print(f"Customer: {customer.name}")

for food in food_items:
    print(f"Food: {food.name} priced at {food.price}")

# 关闭会话
session.close()

Customer: John Doe
Food: Pizza Margherita priced at 8.5
Food: Pizza Salami priced at 9.5
Food: Pizza Quattro Formaggi priced at 10.5


```
sqlite:///customer.db
│     │││
│     ││└── 文件路径（相对路径）
│     │└─── 第3个斜杠：路径开始
│     └──── 第1-2个斜杠：协议分隔符（固定写法）
└────────── 数据库类型
```

In [64]:
from typing import TypedDict
from langchain_core.messages import SystemMessage, BaseMessage

class AgentState(TypedDict):
    question: str
    messages: list[BaseMessage]
    customer_name: str
    tool_calls:list[str]
    order_check: dict[str, str]
    generation: str
    sys_msg: SystemMessage

In [65]:
import os
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0
)

In [66]:
from langchain_core.prompts import ChatPromptTemplate
from  langchain_core.output_parsers import StrOutputParser
import os

system = """You task is to identify items in the question of a User: Identify the following items:

food_items (str): List of food item names. Respond with 'Yes' if the food items are provided and 'No' if they are missing.
delivery_address (str): Delivery address for the order. Respond with 'Yes' if the delivery address is provided and 'No' if it is missing.
order_date (str): Date and time for the order. Respond with 'Yes' if the order date is provided and 'No' if it is missing.
Again: Remember, ONLY answer with 'YES' and 'NO' for each item.

Examples:
"I want to order a pizza Salami" -> 'food_items': 'Yes', 'delivery_address': 'No', 'order_date': 'No'
"I want to order a pizza Salami at 9pm" -> 'food_items': 'Yes', 'delivery_address': 'No', 'order_date': 'Yes'
"I want to order a pizza Salami to 123 Fakestreet, Chicago" -> 'food_items': 'Yes', 'delivery_address': 'Yes', 'order_date': 'No'
"""

order_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}")
    ]   
)

order_checker_llm = order_prompt | llm | StrOutputParser()
order_checker_llm.invoke(
    {"question": "I want to order Pizza Salami to Fake Street 123"}
)

"'food_items': 'Yes', 'delivery_address': 'Yes', 'order_date': 'No'"

In [67]:
system_inform = """Based on the order details provided, inform the user of any missing information.
If the food items are missing, include "Please specify the food items you want to order."
If the delivery address is missing, include "Please provide the delivery address."
If the order date is missing, include "Please provide the date and time for the order."

For example, if both the delivery address and order date are missing, the message should be "Your information is incomplete: Please provide your delivery address and order date."
"""

inform_prompt = ChatPromptTemplate.from_messages([
    ("system", system_inform),
    ("human", "{information}")
])

missing_info_chain = inform_prompt | llm | StrOutputParser()
missing_info_chain.invoke({
    "information": "{'food_items': 'Yes', 'delivery_address': 'Yes', 'order_date': 'No'}"
})

'Your information is incomplete: Please provide the date and time for the order.'

In [68]:
def get_name_from_token(state: str):
    return "John Doe" # fake example

In [69]:
from langchain_core.tools import tool

@tool
def create_order(customer_name: str, food_items: list, delivery_address: str, order_date: str):
    """
    Create a new order for a customer with a list of food items, a delivery address, and an order date.

    Args:
        customer_name (str): Name of the customer placing the order.
        food_items (list): List of food item names.
        delivery_address (str): Delivery address for the order.
        order_date (str): Date and time for the order.

    Returns:
        str: A string containing the details of the latest order.
        str: Error message if the customer or any food item is not found.

    This function interacts with the database to create new orders for the specified customer.    
    """
    print("--- CREATING ORDER ---")
    try:
        customer = session.query(Customer).filter_by(name=customer_name).first()
        if not customer:
            return f"Customer with name {customer_name} not found."
        
        latest_order = None
        order_date = datetime.strptime(order_date, "%Y-%m-%d %H:%M")

        for food_name in food_items:
            food_item = session.query(FoodItem).filter_by(name=food_name).first()
            if not food_item:
                return f"Food item {food_name} not found."
            new_order = Order(
                customer_id=customer.id,
                food_item_id=food_item.id,
                order_date=order_date,
                delivery_address=delivery_address,
            )

            session.add(new_order)
            latest_order = new_order
        
        session.commit()

        return f"Order placed: {customer_name} ordered {food_items} to {delivery_address} at {latest_order.order_date}"
    except Exception as e:
        session.rollback()
        return f"Failed to execute. Error: {repr(e)}"

@tool
def get_all_orders(customer_name: str):
    """
    Retrieve all orders for a specific customer.

    Args:
        customer_name (str): Name of the customer to retrieve orders for.

    Returns:
        str: A string containing the details of all orders for the specified customer.
        str: Error message if the customer is not found.

    This function interacts with the database to retrieve all orders for the specified customer.    
    """
    try:
        customer = session.query(Customer).filter_by(name=customer_name).first()
        if not customer:
            return f"Customer with name {customer_name} not found."
        
        # ✨ 用relationship直接访问！
        if not customer.orders:
            return f"No orders found for customer {customer_name}."
        
        order_details = "\n".join([
            f"Order ID: {order.id}, Food Item: {order.food_item.name}, ..."
            for order in customer.orders  # ✨ 这里也用relationship
        ])
        
        return f"Orders for customer {customer_name}:\n{order_details}"
    except Exception as e:
        return f"Failed to execute. Error: {repr(e)}"  # 删掉rollback

In [70]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import SystemMessagePromptTemplate
from langchain_core.messages import ToolMessage, HumanMessage

template = """You are a service Bot of the Bella Vista restaurant. Be kind and friendly. Always use the Customer's name when you speak to them.
**IMPORTANT GUIDELINES:**
1. When the customer provides ALL required information (food items, delivery address, and order date/time), call the appropriate tool IMMEDIATELY. Do NOT ask for confirmation.
2. Only ask questions when information is MISSING or AMBIGUOUS.
3. You already know the customer's name from the system - use it directly without asking.
4. Be proactive and efficient - customers prefer quick service over unnecessary confirmations.
Customer Name: {customer}
"""
prompt = SystemMessagePromptTemplate.from_template(template)
sys_msg = prompt.format(customer="John Doe")

In [71]:
raw_hu_msg = HumanMessage(content="I want to order a Pizza Salami to the Fake Street 123 for 9:00")

In [72]:
system_time = """Identify and rewrite the time to match the correct format.
If the provided time is not in the format '%Y-%m-%d %H:%M', rewrite the complete question, keep everything unchanged, despite the time"

Today is: {today}

Important: The correct format, take a look at the example:
Example:
User: 'I want to order a Pizza Salami to the Fakestreet 123 for 9:00'
Desired: 'I want to order a Pizza Salami to the Fakestreet 123 for 2024-05-30 09:00'
"""

prosystem_time_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_time),
        ("human", "{question}"),
    ]
)

In [73]:
from datetime import datetime

rewrite_chain = prosystem_time_prompt | llm
rewritten_msg = rewrite_chain.invoke(
    {
        "question": "I want to order a Pizza Salami to the Fakestreet 123 for 9:00",
        "today": str(datetime.today()),
    }
)

In [74]:
rewritten_msg

AIMessage(content='I want to order a Pizza Salami to the Fakestreet 123 for 2025-12-12 09:00', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 159, 'total_tokens': 186, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 159}, 'model_provider': 'openai', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_eaab8d114b_prod0820_fp8_kvcache', 'id': 'd3f9a9f7-32b6-4f53-b27a-09ddf2052b2e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b1091-afa1-7ba1-ad79-0b111f1d7fc1-0', usage_metadata={'input_tokens': 159, 'output_tokens': 27, 'total_tokens': 186, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}})

In [75]:
messages = [sys_msg, HumanMessage(content=rewritten_msg.content)]
messages

[SystemMessage(content="You are a service Bot of the Bella Vista restaurant. Be kind and friendly. Always use the Customer's name when you speak to them.\n**IMPORTANT GUIDELINES:**\n1. When the customer provides ALL required information (food items, delivery address, and order date/time), call the appropriate tool IMMEDIATELY. Do NOT ask for confirmation.\n2. Only ask questions when information is MISSING or AMBIGUOUS.\n3. You already know the customer's name from the system - use it directly without asking.\n4. Be proactive and efficient - customers prefer quick service over unnecessary confirmations.\nCustomer Name: John Doe\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I want to order a Pizza Salami to the Fakestreet 123 for 2025-12-12 09:00', additional_kwargs={}, response_metadata={})]

In [76]:
model_with_tools = llm.bind_tools([create_order, get_all_orders])

In [77]:
ai_msg = model_with_tools.invoke(messages)

In [78]:
messages.append(ai_msg)

In [79]:
messages

[SystemMessage(content="You are a service Bot of the Bella Vista restaurant. Be kind and friendly. Always use the Customer's name when you speak to them.\n**IMPORTANT GUIDELINES:**\n1. When the customer provides ALL required information (food items, delivery address, and order date/time), call the appropriate tool IMMEDIATELY. Do NOT ask for confirmation.\n2. Only ask questions when information is MISSING or AMBIGUOUS.\n3. You already know the customer's name from the system - use it directly without asking.\n4. Be proactive and efficient - customers prefer quick service over unnecessary confirmations.\nCustomer Name: John Doe\n", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I want to order a Pizza Salami to the Fakestreet 123 for 2025-12-12 09:00', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Hello John! I can see you'd like to order a Pizza Salami for delivery to Fakestreet 123 on December 12th, 2025 at 9:00 AM.\n\nI have all the information

In [80]:
ai_msg.tool_calls

[{'name': 'create_order',
  'args': {'customer_name': 'John Doe',
   'food_items': ['Pizza Salami'],
   'delivery_address': 'Fakestreet 123',
   'order_date': '2025-12-12 09:00'},
  'id': 'call_00_RtcyHVoaPX9OLPljaGqAqFhu',
  'type': 'tool_call'}]

In [81]:
for tool_call in ai_msg.tool_calls:
    print("Use Tool:", tool_call)
    selected_tool = {"create_order": create_order, "get_all_orders": get_all_orders}[
        tool_call["name"].lower()
    ]
    tool_output = selected_tool.invoke(tool_call["args"])
    print(tool_output)
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

Use Tool: {'name': 'create_order', 'args': {'customer_name': 'John Doe', 'food_items': ['Pizza Salami'], 'delivery_address': 'Fakestreet 123', 'order_date': '2025-12-12 09:00'}, 'id': 'call_00_RtcyHVoaPX9OLPljaGqAqFhu', 'type': 'tool_call'}
--- CREATING ORDER ---
Order placed: John Doe ordered ['Pizza Salami'] to Fakestreet 123 at 2025-12-12 09:00:00


In [89]:
orders = session.query(Order).all()

for order in orders:
    print(f"Order: {order.customer.name} ordered {order.food_item.name}")

Order: John Doe ordered Pizza Salami
